In [0]:
CATALOG ="flights_delay_pipeline"
SCHEMA = "flights_schema"

SILVER = f"{CATALOG}.{SCHEMA}.silver_flights"
GOLD_AIRLINE     = f"{CATALOG}.{SCHEMA}.gold_airline_performance"
GOLD_AIRPORT     = f"{CATALOG}.{SCHEMA}.gold_airport_traffic"
GOLD_MONTHLY     = f"{CATALOG}.{SCHEMA}.gold_monthly_delays"
GOLD_CANCEL      = f"{CATALOG}.{SCHEMA}.gold_cancellation_reasons"
GOLD_ROUTE       = f"{CATALOG}.{SCHEMA}.gold_route_performance"
GOLD_HOURLY   = f"{CATALOG}.{SCHEMA}.gold_hourly_delays"

print(SILVER)
print(GOLD_AIRLINE)

flights_delay_pipeline.flights_schema.silver_flights
flights_delay_pipeline.flights_schema.gold_airline_performance


In [0]:
from pyspark.sql.functions import (
    col, count, avg, sum, round, when, desc,
    month, date_format, countDistinct
)
silver = spark.table("flights_delay_pipeline.flights_schema.silver_flights")
silver.printSchema()
silver.count()

root
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- YEAR: long (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- DAY: long (nullable = true)
 |-- DAY_OF_WEEK: long (nullable = true)
 |-- FLIGHT_NUMBER: long (nullable = true)
 |-- SCHEDULED_DEPARTURE: long (nullable = true)
 |-- DEPARTURE_TIME: long (nullable = true)
 |-- DEPARTURE_DELAY: long (nullable = true)
 |-- TAXI_OUT: long (nullable = true)
 |-- AIR_TIME: long (nullable = true)
 |-- DISTANCE: long (nullable = true)
 |-- TAXI_IN: long (nullable = true)
 |-- SCHEDULED_ARRIVAL: long (nullable = true)
 |-- ARRIVAL_TIME: long (nullable = true)
 |-- ARRIVAL_DELAY: long (nullable = true)
 |-- DIVERTED: boolean (nullable = true)
 |-- CANCELLED: boolean (nullable = true)
 |-- CANCELLATION_REASON: string (nullable = true)
 |-- AIR_SYSTEM_DELAY: long (nullable = true)
 |-- SECURITY_DELAY: long (nullable = true)
 |-- AIRLINE_DELAY: long (nu

5332914

In [0]:
gold_airline_df = (
    silver
    .filter(col("CANCELLED") == False)          # only flights that flew
    .groupBy("AIRLINE", "AIRLINE_NAME")
    .agg(
        count("*").alias("total_flights"),
        round(avg("DEPARTURE_DELAY"), 2).alias("avg_departure_delay"),
        round(avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay"),
        sum(when(col("DEPARTURE_DELAY") > 0, 1).otherwise(0)).alias("delayed_flights"),
    )
    .withColumn(
        "on_time_pct",
        round(
            (col("total_flights") - col("delayed_flights")) * 100.0 / col("total_flights"),
            2
        )
    )
    .orderBy(desc("total_flights"))
)
display(gold_airline_df)

AIRLINE,AIRLINE_NAME,total_flights,avg_departure_delay,avg_arrival_delay,delayed_flights,on_time_pct
WN,Southwest Airlines Co.,1141618,10.98,4.84,527083,53.83
DL,Delta Air Lines Inc.,796522,7.75,0.68,263384,66.93
AA,American Airlines Inc.,638500,9.29,3.94,223646,64.97
OO,Skywest Airlines Inc.,529826,8.15,6.22,159905,69.82
EV,Atlantic Southeast Airlines,511495,9.02,6.96,158652,68.98
UA,United Air Lines Inc.,463384,15.07,6.21,239143,48.39
MQ,American Eagle Airlines Inc.,257898,10.57,7.32,88791,65.57
B6,JetBlue Airways,240989,11.81,6.95,93672,61.13
US,US Airways Inc.,194648,6.11,3.71,62452,67.92
AS,Alaska Airlines Inc.,157418,1.95,-0.79,40382,74.35


In [0]:
(
    gold_airline_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_AIRLINE)
)
print(f" Saved: {GOLD_AIRLINE}")

 Saved: flights_delay_pipeline.flights_schema.gold_airline_performance


In [0]:
origins = silver.select(
    col("ORIGIN_AIRPORT").alias("airport"),
    col("ORIGIN_CITY").alias("city"),
    col("ORIGIN_STATE").alias("state"),
    col("DEPARTURE_DELAY").alias("delay"),
    col("CANCELLED"),
)

destinations = silver.select(
    col("DESTINATION_AIRPORT").alias("airport"),
    col("DESTINATION_CITY").alias("city"),
    col("DESTINATION_STATE").alias("state"),
    col("ARRIVAL_DELAY").alias("delay"),
    col("CANCELLED"),
)

airport_events = origins.unionByName(destinations)
print(f"Total airport events: {airport_events.count():,}")

Total airport events: 10,665,828


In [0]:
gold_airport_df = (
    airport_events
    .groupBy("airport", "city", "state")
    .agg(
        count("*").alias("total_flights"),
        round(avg("delay"), 2).alias("avg_delay"),
        sum(when(col("CANCELLED") == True, 1).otherwise(0)).alias("cancellations"),
    )
    .withColumn(
        "cancellation_rate_pct",
        round(col("cancellations") * 100.0 / col("total_flights"), 2)
    )
    .orderBy(desc("total_flights"))
)

display(gold_airport_df)

airport,city,state,total_flights,avg_delay,cancellations,cancellation_rate_pct
ATL,Atlanta,GA,693740,5.79,5272,0.76
ORD,Chicago,IL,571790,10.71,17821,3.12
DFW,Dallas-Fort Worth,TX,479133,8.71,13003,2.71
DEN,Denver,CO,392065,8.47,4433,1.13
LAX,Los Angeles,CA,389369,8.39,4423,1.14
SFO,San Francisco,CA,295974,8.89,4453,1.5
PHX,Phoenix,AZ,293627,5.8,2112,0.72
IAH,Houston,TX,293305,8.63,4562,1.56
LAS,Las Vegas,NV,266379,7.28,1796,0.67
MSP,Minneapolis,MN,224245,5.32,1666,0.74


In [0]:
(
    gold_airport_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_AIRPORT)
)
print(f" Saved: {GOLD_AIRPORT}")

 Saved: flights_delay_pipeline.flights_schema.gold_airport_traffic


In [0]:
gold_monthly_df = (
    silver
    .filter(col("CANCELLED") == False)
    .groupBy("YEAR", "MONTH")
    .agg(
        count("*").alias("total_flights"),
        round(avg("DEPARTURE_DELAY"), 2).alias("avg_departure_delay"),
        round(avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay"),
        sum(when(col("DEPARTURE_DELAY") > 15, 1).otherwise(0)).alias("major_delays"),
    )
    .orderBy("YEAR", "MONTH")
)
display(gold_monthly_df)

YEAR,MONTH,total_flights,avg_departure_delay,avg_arrival_delay,major_delays
2015,1,457986,9.73,5.81,87557
2015,2,408674,11.81,8.32,87806
2015,3,493310,9.63,4.92,91651
2015,4,480631,7.68,3.16,76484
2015,5,491299,9.42,4.49,86416
2015,6,494777,13.94,9.6,112834
2015,7,515912,11.37,6.43,105962
2015,8,505484,9.91,4.61,92838
2015,9,462871,4.81,-0.77,56003
2015,11,463373,6.92,1.1,66872


In [0]:
(
    gold_monthly_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_MONTHLY)
)
print(f" Saved: {GOLD_MONTHLY}")

 Saved: flights_delay_pipeline.flights_schema.gold_monthly_delays


In [0]:
total_cancelled = silver.filter(col("CANCELLED") == True).count()

gold_cancel_df = (
    silver
    .filter(col("CANCELLED") == True)
    .groupBy("CANCELLATION_REASON")
    .agg(count("*").alias("cancellations"))
    .withColumn(
        "pct_of_total_cancellations",
        round(col("cancellations") * 100.0 / total_cancelled, 2)
    )
    .orderBy(desc("cancellations"))
)
display(gold_cancel_df)

CANCELLATION_REASON,cancellations,pct_of_total_cancellations
Weather,47874,54.76
Carrier,24309,27.8
National Airspace System,15225,17.41
Security,22,0.03


In [0]:
(
    gold_cancel_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CANCEL)
)
print(f" Saved: {GOLD_CANCEL}")

 Saved: flights_delay_pipeline.flights_schema.gold_cancellation_reasons


In [0]:
gold_route_df = (
    silver
    .filter(col("CANCELLED") == False)
    .groupBy(
        "ORIGIN_AIRPORT", "ORIGIN_CITY",
        "DESTINATION_AIRPORT", "DESTINATION_CITY"
    )
    .agg(
        count("*").alias("flights"),
        round(avg("DEPARTURE_DELAY"), 2).alias("avg_departure_delay"),
        round(avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay"),
    )
    .filter(col("flights") >= 100)                  # ← important: exclude tiny routes
    .orderBy(desc("avg_departure_delay"))
)

display(gold_route_df)

ORIGIN_AIRPORT,ORIGIN_CITY,DESTINATION_AIRPORT,DESTINATION_CITY,flights,avg_departure_delay,avg_arrival_delay
JFK,New York,EGE,Eagle,107,46.98,38.01
ASE,Aspen,DFW,Dallas-Fort Worth,279,39.84,39.97
ACY,Atlantic City,DTW,Detroit,145,37.39,30.9
EWR,Newark,PDX,Portland,320,33.64,14.61
SRQ,Sarasota,LGA,New York,423,33.48,30.17
BQN,Aguadilla,EWR,Newark,287,33.39,23.84
MVY,Marthas Vineyard,JFK,New York,121,33.32,18.31
ACY,Atlantic City,ORD,Chicago,141,33.1,24.96
PIT,Pittsburgh,JFK,New York,161,32.25,27.34
EGE,Eagle,EWR,Newark,106,31.56,17.08


In [0]:
(
    gold_route_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ROUTE)
)
print(f" Saved: {GOLD_ROUTE}")

 Saved: flights_delay_pipeline.flights_schema.gold_route_performance


In [0]:
# Preview the conversion before aggregating
from pyspark.sql.functions import floor
sample = (
    silver
    .filter(col("CANCELLED") == False)
    .select(
        "SCHEDULED_DEPARTURE",
        floor(col("SCHEDULED_DEPARTURE") / 100).cast("int").alias("departure_hour")
    )
    .distinct()
    .orderBy("SCHEDULED_DEPARTURE")
)

display(sample)

SCHEDULED_DEPARTURE,departure_hour
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,0
10,0


In [0]:
gold_hourly_df = (
    silver
    .filter(col("CANCELLED") == False)
    .withColumn(
        "departure_hour",
        floor(col("SCHEDULED_DEPARTURE") / 100).cast("int")
    )
    .groupBy("departure_hour")
    .agg(
        count("*").alias("total_flights"),
        round(avg("DEPARTURE_DELAY"), 2).alias("avg_departure_delay"),
        round(avg("ARRIVAL_DELAY"), 2).alias("avg_arrival_delay"),
        sum(when(col("DEPARTURE_DELAY") > 15, 1).otherwise(0)).alias("major_delays"),
    )
    .orderBy("departure_hour")
)

display(gold_hourly_df)

departure_hour,total_flights,avg_departure_delay,avg_arrival_delay,major_delays
0,13598,7.45,1.09,2251
1,4867,8.33,4.0,833
2,1315,7.14,1.62,223
3,719,9.48,1.82,150
4,525,9.29,2.65,123
5,106656,2.02,-3.63,6105
6,366176,2.24,-2.42,23923
7,354591,3.43,-1.21,30990
8,344116,4.89,0.14,37303
9,317081,5.92,1.3,40864


In [0]:
(
    gold_hourly_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_HOURLY)
)
print(f" Saved: {GOLD_HOURLY}")

 Saved: flights_delay_pipeline.flights_schema.gold_hourly_delays


In [0]:
gold_tables = [
    GOLD_AIRLINE,
    GOLD_AIRPORT,
    GOLD_MONTHLY,
    GOLD_CANCEL,
    GOLD_ROUTE,
    GOLD_HOURLY,
]

print("Gold layer summary")
print("=" * 65)
for t in gold_tables:
    name = t.split(".")[-1]
    n = spark.table(t).count()
    print(f"  {name:<35} {n:>8,} rows")

Gold layer summary
  gold_airline_performance                  14 rows
  gold_airport_traffic                     322 rows
  gold_monthly_delays                       11 rows
  gold_cancellation_reasons                  4 rows
  gold_route_performance                 4,078 rows
  gold_hourly_delays                        24 rows
